# Flax Basics

Flax extend JAX by providing deep learning primitives without violating functional purity. While a PyTorch nn.Module holds both the architectural definition and its mutable weights inside self, a Flax nn.Module acts strictly as a **stateless blueprint** for tensor transformations.

Paramaeter ownership, optimization updates, and execution passes are separated into three distinct phases: **Module Definition**, **Initialization**, and **Application**

In [1]:
import jax
import jax.numpy as jnp

from flax import linen as nn

## Defining Layers

The standard way to define networks in flax is using flax.linen (imported conventionally as nn). The @nn.compact decorator allows you to define sub-layers inline directly inside the forward pass function (__call__), eliminating boilerplate: (This feels a lot like nn.Sequential in PyTorch).

In [6]:
class SimpleMLP(nn.Module):
    hidden_dim: int
    output_dim: int

    @nn.compact
    def __call__(self, x):
        # Sub-layeers are defined inline; Flax handles parameter instatiation
        x = nn.Dense(features=self.hidden_dim)(x)
        x = nn.relu(x)
        x = nn.Dense(features=self.output_dim)(x)
        return x

## The Functional Split
Instantiating model = SimpleMLP() creates an abstract template, not an initalized model with weights in memory.
- init(rng, dummy_input): Traces the module with a sample tensor and random PRNG key to infer tensor shapes and allocate weight matrices. It returns an immutalbe dictory (FrozenDict) of parameters
- apply(variables, x): Evaluates the network using explicity provided parameter weights.

In [7]:
model = SimpleMLP(hidden_dim=32, output_dim=1)
key = jax.random.key(0)
dummy_x = jnp.ones((1,8)) # Batch Size 1, Feature Inputs 8

# Initialize Weights:
variables = model.init(key, dummy_x)
params = variables['params'] # Contains nested weight and bias dicts for sub-layers

# Forward Pass:
predictions = model.apply({'params': params}, dummy_x)

print(f"The Dummy Date is: {dummy_x}")
print(f"The Dummy forward pass is: {predictions}")

The Dummy Date is: [[1. 1. 1. 1. 1. 1. 1. 1.]]
The Dummy forward pass is: [[-0.45136154]]


Continue Here: https://share.gemini.google/1KqywRM2ixRP